In [ ]:
import os
import yaml
import shutil
from pathlib import Path
from ultralytics import YOLO
from roboflow import Roboflow

cwd = Path.cwd()
model_dir = cwd if cwd.name == "model" else (cwd / "model" if (cwd / "model").exists() else cwd)
model_dir = model_dir.resolve()
combined_dir = model_dir / "combined_dishes"

In [ ]:
rf = Roboflow(api_key="redacted")

datasets_to_download = [
    {"tag": "custom", "ws": "sinksnitch", "proj": "my-first-project-gkj5j", "ver": 4},
    {"tag": "dirty", "ws": "anton-althoff", "proj": "dirty-kitchen", "ver": 2},
    {"tag": "yolo_obyci", "ws": "yolo-obyci", "proj": "final_dataset-akpsj", "ver": 7},
    {"tag": "dishwasher", "ws": "provisioning", "proj": "dishwasher-status", "ver": 2},
    {"tag": "disheslabels", "ws": "disheslabels", "proj": "dishes-detection-fmfvy", "ver": 1},
    {"tag": "venkiboo", "ws": "venkiboo", "proj": "object-detection-yolo-v9-brnzq", "ver": 1},
]

local_datasets = []
for ds in datasets_to_download:
    proj = rf.workspace(ds["ws"]).project(ds["proj"])
    version = proj.version(ds["ver"])
    downloaded = version.download("yolov8")
    src = Path(downloaded.location).resolve()
    dst = (model_dir / src.name).resolve()
    if src != dst:
        if dst.exists():
            shutil.rmtree(dst)
        shutil.move(str(src), str(dst))
    local_datasets.append({"tag": ds["tag"], "path": dst})

for ds in local_datasets:
    train_count = len(list((ds["path"] / "train" / "images").iterdir())) if (ds["path"] / "train" / "images").exists() else 0
    val_count = len(list((ds["path"] / "valid" / "images").iterdir())) if (ds["path"] / "valid" / "images").exists() else 0
    test_count = len(list((ds["path"] / "test" / "images").iterdir())) if (ds["path"] / "test" / "images").exists() else 0
    total = train_count + val_count + test_count
    print(f"{ds['tag']:8s}: {train_count:4d} train, {val_count:4d} val, {test_count:4d} test = {total:4d} total")

for sub in ["train/images", "train/labels", "valid/images", "valid/labels", "test/images", "test/labels"]:
    (combined_dir / sub).mkdir(parents=True, exist_ok=True)

def copy_split(ds_tag, ds_path, split):
    img_src = ds_path / split / "images"
    lbl_src = ds_path / split / "labels"
    img_dst = combined_dir / split / "images"
    lbl_dst = combined_dir / split / "labels"
    if not img_src.exists():
        return
    for img_file in img_src.iterdir():
        if img_file.suffix.lower() not in [".jpg", ".jpeg", ".png"]:
            continue
        stem = img_file.stem
        lbl_file = lbl_src / f"{stem}.txt"
        new_stem = f"{ds_tag}_{stem}"
        shutil.copy2(img_file, img_dst / f"{new_stem}{img_file.suffix}")
        if lbl_file.exists():
            with open(lbl_file, "r") as f:
                lines = f.readlines()
            new_lines = []
            for line in lines:
                parts = line.strip().split()
                if len(parts) >= 5:
                    parts[0] = "0"
                    new_lines.append(" ".join(parts) + "\n")
            with open(lbl_dst / f"{new_stem}.txt", "w") as f:
                f.writelines(new_lines)

for ds in local_datasets:
    for split in ["train", "valid", "test"]:
        copy_split(ds["tag"], ds["path"], split)

data_yaml_path = combined_dir / "data.yaml"
with open(data_yaml_path, "w") as f:
    yaml.safe_dump({
        "path": str(combined_dir.resolve()),
        "train": "train/images",
        "val": "valid/images",
        "test": "test/images",
        "nc": 1,
        "names": ["dish"],
    }, f)

train_count = len(list((combined_dir / "train" / "images").iterdir())) if (combined_dir / "train" / "images").exists() else 0
val_count = len(list((combined_dir / "valid" / "images").iterdir())) if (combined_dir / "valid" / "images").exists() else 0
test_count = len(list((combined_dir / "test" / "images").iterdir())) if (combined_dir / "test" / "images").exists() else 0

for ds in local_datasets:
    if ds["path"].exists() and ds["path"] != combined_dir:
        shutil.rmtree(ds["path"])

In [ ]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None

model = YOLO('yolov8s.pt')

results = model.train(
    data=str(combined_dir / 'data.yaml'),
    epochs=50,
    imgsz=640,
    batch=6,
    patience=10,
    device=0,
    project=str(model_dir),
    name='training',
    exist_ok=True,
    lr0=0.001,
    lrf=0.01,
    optimizer='AdamW',
    degrees=15.0,
    mixup=0.2,
    workers=0,
    cache=False,
    amp=True,
    plots=False,
    save_json=False
)

In [ ]:
best_model = YOLO(str(model_dir / 'training' / 'weights' / 'best.pt'))
shutil.copy(str(model_dir / 'training' / 'weights' / 'best.pt'), str(model_dir / 'dish_detector.pt'))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import cv2
import random

plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

results_csv = model_dir / 'training' / 'results.csv'
output_dir = model_dir / 'results'
output_dir.mkdir(exist_ok=True)

df = pd.read_csv(results_csv)

fig, ax = plt.subplots(figsize=(10, 6))

total_train = df['train/box_loss'] + df['train/cls_loss'] + df['train/dfl_loss']
total_val = df['val/box_loss'] + df['val/cls_loss'] + df['val/dfl_loss']

ax.plot(df['epoch'], total_train, label='Train Total Loss', linewidth=2.5, color='#2E86AB')
ax.plot(df['epoch'], total_val, label='Validation Total Loss', linewidth=2.5, color='#A23B72', linestyle='--')

ax.set_xlabel('Epoch', fontsize=13, fontweight='bold')
ax.set_ylabel('Total Loss', fontsize=13, fontweight='bold')
ax.set_title('Training and Validation Loss Over Epochs', fontsize=15, fontweight='bold', pad=15)
ax.legend(fontsize=11, framealpha=0.9)
ax.grid(True, alpha=0.3)
ax.tick_params(labelsize=11)

plt.tight_layout()
plt.savefig(output_dir / 'figure1_total_loss.png', dpi=300, bbox_inches='tight')
plt.close()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Model Performance Metrics Over Training', fontsize=16, fontweight='bold', y=0.995)

axes[0, 0].plot(df['epoch'], df['metrics/mAP50(B)'], label='mAP@50', linewidth=2.5, color='#06A77D')
axes[0, 0].set_xlabel('Epoch', fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel('mAP@50', fontsize=12, fontweight='bold')
axes[0, 0].set_title('Mean Average Precision @ IoU=0.50', fontsize=13, fontweight='bold')
axes[0, 0].legend(fontsize=10, framealpha=0.9)
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].set_ylim([0, 1])
axes[0, 0].tick_params(labelsize=10)

axes[0, 1].plot(df['epoch'], df['metrics/mAP50-95(B)'], label='mAP@50-95', linewidth=2.5, color='#F18F01')
axes[0, 1].set_xlabel('Epoch', fontsize=12, fontweight='bold')
axes[0, 1].set_ylabel('mAP@50-95', fontsize=12, fontweight='bold')
axes[0, 1].set_title('Mean Average Precision @ IoU=0.50-0.95', fontsize=13, fontweight='bold')
axes[0, 1].legend(fontsize=10, framealpha=0.9)
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].set_ylim([0, 1])
axes[0, 1].tick_params(labelsize=10)

axes[1, 0].plot(df['epoch'], df['metrics/precision(B)'], label='Precision', linewidth=2.5, color='#C73E1D')
axes[1, 0].set_xlabel('Epoch', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Precision', fontsize=12, fontweight='bold')
axes[1, 0].set_title('Precision', fontsize=13, fontweight='bold')
axes[1, 0].legend(fontsize=10, framealpha=0.9)
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].set_ylim([0, 1])
axes[1, 0].tick_params(labelsize=10)

axes[1, 1].plot(df['epoch'], df['metrics/recall(B)'], label='Recall', linewidth=2.5, color='#6A4C93')
axes[1, 1].set_xlabel('Epoch', fontsize=12, fontweight='bold')
axes[1, 1].set_ylabel('Recall', fontsize=12, fontweight='bold')
axes[1, 1].set_title('Recall', fontsize=13, fontweight='bold')
axes[1, 1].legend(fontsize=10, framealpha=0.9)
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].set_ylim([0, 1])
axes[1, 1].tick_params(labelsize=10)

plt.tight_layout()
plt.savefig(output_dir / 'figure2_performance_metrics.png', dpi=300, bbox_inches='tight')
plt.close()

test_images_dir = combined_dir / 'test' / 'images'
all_test_images = list(test_images_dir.glob('*.jpg')) + list(test_images_dir.glob('*.jpeg')) + list(test_images_dir.glob('*.png'))
random.shuffle(all_test_images)
test_images = all_test_images[:12]

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
fig.suptitle('Sample Detection Results on Test Set', fontsize=16, fontweight='bold', y=0.995)

axes = axes.flatten()

for idx, img_path in enumerate(test_images):
    results = best_model(str(img_path), conf=0.25, verbose=False)
    annotated = results[0].plot()
    annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    
    axes[idx].imshow(annotated_rgb)
    axes[idx].axis('off')
    axes[idx].set_title(f'Detections: {len(results[0].boxes)}', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig(output_dir / 'figure3_sample_detections.png', dpi=300, bbox_inches='tight')
plt.close()

validation_metrics = best_model.val(project=str(model_dir), name='validation', plots=False, save_json=False, exist_ok=True)
test_metrics = best_model.val(data=str(combined_dir / 'data.yaml'), split='test', project=str(model_dir), name='test', plots=False, save_json=False, exist_ok=True)

for item in model_dir.iterdir():
    if item.is_dir():
        if item.name == 'validation' or item.name == 'test' or \
           (item.name.startswith('validation') and item.name != 'validation') or \
           (item.name.startswith('test') and item.name != 'test' and len(item.name) > 4):
            try:
                shutil.rmtree(item)
            except:
                pass

repo_root = model_dir.parent
runs_detect = repo_root / 'runs' / 'detect'
if runs_detect.exists():
    try:
        shutil.rmtree(runs_detect)
        if (repo_root / 'runs').exists() and not any((repo_root / 'runs').iterdir()):
            (repo_root / 'runs').rmdir()
    except:
        pass

summary_lines = [
    "",
    "Validation Set:",
    f"  mAP@50:    {validation_metrics.box.map50:.4f}",
    f"  mAP@50-95: {validation_metrics.box.map:.4f}",
    f"  Precision: {validation_metrics.box.mp:.4f}",
    f"  Recall:    {validation_metrics.box.mr:.4f}",
    "",
    "Test Set:",
    f"  mAP@50:    {test_metrics.box.map50:.4f}",
    f"  mAP@50-95: {test_metrics.box.map:.4f}",
    f"  Precision: {test_metrics.box.mp:.4f}",
    f"  Recall:    {test_metrics.box.mr:.4f}",
    ""
]

summary_text = "\n".join(summary_lines)
summary_file = output_dir / 'performance_summary.txt'
with open(summary_file, 'w') as f:
    f.write(summary_text)